# Sesión 7: Pruebas de hipótesis y bootstrapping con la ENDI

En este notebook vas a replicar el flujo completo de análisis trabajado en clase, pero usando dos variables distintas:

| Variable | Tipo | Descripción |
|---|---|---|
| `ane6_59_new` | Discreta (0/1) | Anemia en niños de 6 a 59 meses |
| `f1_s5_4_1` | Continua | Peso en kg (primera toma) |

Y las cruzamos con: `area` y `sexo`.

El objetivo es que construyas el notebook desde cero usando el de clase como referencia. Cada celda tiene un comentario guía que indica qué debes hacer.

## Bloque 0: Preparación del entorno

Antes de ejecutar este notebook, asegúrate de activar el entorno virtual desde la carpeta raíz del repositorio.

```
cd <ruta/a carpeta/de trabajo>
.\venv\Scripts\activate
```

Si ya tenías una versión anterior del `requirements.txt` instalada, solo necesitas agregar el paquete nuevo:
```
pip install svy
```

Una vez activado, instala todas las dependencias con:
```
pip install -r requirements.txt
```

Si no tienes el archivo `requirements.txt`, instala los paquetes manualmente:
```
pip install polars pyreadr scipy svy
```

### Ejercicio 0.1: Importar librerías

In [ ]:
# Importa polars con alias pl
# Importa polars.selectors con alias cs
# Importa pyreadr
# Importa numpy con alias np
# Importa scipy.stats con alias stats
# Importa svy

### Ejercicio 0.2: Carga y selección de variables

Carga la tabla de personas desde el archivo `.rds` y selecciona las variables que necesitas para este ejercicio.

Las variables que debes seleccionar son:

| Variable original | Nombre nuevo | Descripción |
|---|---|---|
| `id_upm` | - | Identificador de unidad primaria de muestreo |
| `estrato` | - | Estrato muestral |
| `fexp` | - | Factor de expansión normalizado |
| `area` | - | Área (1 = urbano, 2 = rural) |
| `f1_s1_2` | `sexo` | Sexo del niño o niña |
| `edaddias` | - | Edad en días |
| `f1_s5_4_1` | `peso` | Peso en kg (primera toma) |
| `ane6_59_new` | - | Anemia en niños de 6 a 59 meses (0/1) |

In [ ]:
# Define la ruta al archivo .rds en: ..\..\Materiales\Insumos\ENDI\BDD_ENDI_R2_rds\BDD_ENDI_R2_f1_personas.rds
# Lee el archivo con pyreadr y convierte a polars con pl.from_pandas()
# Selecciona las variables indicadas en la tabla de arriba
# Renombra f1_s1_2 -> sexo y f1_s5_4_1 -> peso
# Muestra las primeras 3 filas

### Ejercicio 0.3: Filtro de población de análisis

Filtra los registros que cumplen estas dos condiciones:

1. El niño o niña tiene entre 180 y 1826 días de edad (entre 6 meses y 5 años, que es la población objetivo de `ane6_59_new`).
2. La variable `ane6_59_new` no es nula.

In [ ]:
# Filtra edaddias >= 180 y edaddias < 1826
# Filtra ane6_59_new no nulo
# Imprime el shape resultante
# Muestra las primeras 3 filas

### Ejercicio 0.4: Recodificación de etiquetas

Convierte las variables categóricas a etiquetas legibles con `pl.when()`.

Los códigos son:

- **área**: 1 = urbano, 2 = rural
- **sexo**: 1 = hombre, 2 = mujer

In [ ]:
# Recodifica area: 1 -> "urbano", 2 -> "rural"
# Recodifica sexo: 1 -> "hombre", 2 -> "mujer"
# Muestra las primeras 3 filas

### Ejercicio 0.5: Declaración del diseño muestral

Declara el diseño muestral de la ENDI usando `svy.Design` y crea el objeto `svy.Sample`.

| Parámetro | Variable |
|---|---|
| stratum | `estrato` |
| psu | `id_upm` |
| wgt | `fexp` |

In [ ]:
# Crea el objeto diseno con svy.Design usando stratum, psu y wgt
# Crea el objeto muestra con svy.Sample y aplica set_design()
# Imprime muestra para verificar el diseño declarado

## Bloque 1: Estimaciones con diseño muestral

Antes de hacer pruebas de hipótesis, calculamos las estimaciones puntuales de nuestras variables de interés incorporando el diseño muestral.

### Ejercicio 1.1: Media de peso por area

Estima la media de `peso` por `area` usando `muestra.estimation.mean()`.

In [ ]:
# Estima la media de peso por area con diseño muestral
# Imprime el resultado

### Ejercicio 1.2: Proporción de anemia por area

Estima la proporción de `ane6_59_new` por `area` usando `muestra.estimation.prop()`.

In [ ]:
# Estima la proporción de ane6_59_new por area con diseño muestral
# Imprime el resultado

## Bloque 2: T-test diferencia de peso por sexo

Queremos saber si el peso medio de los niños difiere del de las niñas.

- $H_0$: $\mu_{\text{hombre}} = \mu_{\text{mujer}}$
- $H_1$: $\mu_{\text{hombre}} \neq \mu_{\text{mujer}}$

### Ejercicio 2.1: T-test sin diseño muestral

Separa los datos en dos grupos por sexo y aplica `stats.ttest_ind()` con `equal_var=False` (Welch).

In [ ]:
# Filtra y extrae el array de peso para hombres (drop_nulls + to_numpy)
# Filtra y extrae el array de peso para mujeres (drop_nulls + to_numpy)
# Aplica stats.ttest_ind con equal_var=False
# Imprime la media de cada grupo, el estadístico t y el p-valor

### Ejercicio 2.2: T-test con diseño muestral (test de Wald)

Estima la media de `peso` por `sexo` con `svy` y construye el estadístico de Wald:

$$t_{\text{diseño}} = \frac{\hat{\mu}_1 - \hat{\mu}_2}{\sqrt{\widehat{SE}(\hat{\mu}_1)^2 + \widehat{SE}(\hat{\mu}_2)^2}}$$

Los grados de libertad se aproximan como `n_strata - 1`.

In [ ]:
# Estima la media de peso por sexo con muestra.estimation.mean()
# Extrae est y se para cada grupo desde result.estimates
# Calcula la diferencia, el SE de la diferencia y el estadístico t de Wald
# Calcula el p-valor con stats.t.sf() usando n_strata - 1 grados de libertad
# Imprime la diferencia, t de Wald y p-valor

## Bloque 3: Test de Wald para proporciones -- anemia por area

Queremos saber si la prevalencia de anemia difiere entre el área urbana y rural.

- $H_0$: $p_{\text{urbano}} = p_{\text{rural}}$
- $H_1$: $p_{\text{urbano}} \neq p_{\text{rural}}$

$$t = \frac{\hat{p}_1 - \hat{p}_2}{\sqrt{\widehat{SE}(\hat{p}_1)^2 + \widehat{SE}(\hat{p}_2)^2}}$$

### Ejercicio 3.1: Test de Wald con diseño muestral

Usa la estimación de proporciones del ejercicio 1.2 para construir el test de Wald para la diferencia de proporciones entre áreas.

In [ ]:
# Recupera las estimaciones de ane6_59_new == 1 por area desde est_area.estimates
# Extrae est y se para urbano y rural
# Calcula la diferencia, el SE de la diferencia y el estadístico t de Wald
# Calcula el p-valor con stats.t.sf() usando n_strata - 1 grados de libertad
# Imprime la proporción de cada área, la diferencia, t de Wald y p-valor


## Preguntas de reflexión

Responde en celdas de markdown debajo de cada pregunta.

1. En el ejercicio 2, ¿el t-test sin diseño y el test de Wald llevan a la misma conclusión? ¿El error estándar es mayor o menor cuando incorporas el diseño muestral?

2. En el ejercicio 3, ¿existe diferencia significativa en la prevalencia de anemia entre áreas? ¿En qué área es mayor?

3. ¿Por qué filtramos `edaddias >= 180` para `ane6_59_new` y no usamos el mismo filtro que para `dcronica`?